# 03 · Features and models

All learned transformations fit May. June decides which additions survive and which model wins. Running this notebook in a fresh clone builds the frozen artifacts; rerunning an existing completed study reads its reports.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image
from taxi_tip.config import ROOT, REPORTS, ARTIFACTS
from taxi_tip.data import read_split, clean_trips
print("Project:", ROOT)


In [ ]:
from taxi_tip.workflow import develop
if not (ARTIFACTS / "selection.json").exists():
    selection = develop()
else:
    selection = json.loads((ARTIFACTS / "selection.json").read_text())
    print("Using this run's frozen selection; no model fitting.")
display(selection)

In [ ]:
from taxi_tip.config import FEATURE_GROUPS, FORBIDDEN
from taxi_tip.models import feature_columns, make_pipeline
numeric, categorical = feature_columns(selection["feature_groups"])
assert not FORBIDDEN.intersection(numeric + categorical)
display(pd.DataFrame({"feature": numeric + categorical, "kind": ["numeric"]*len(numeric) + ["categorical"]*len(categorical)}))
display(make_pipeline(selection["feature_groups"]))

In [ ]:
display(pd.read_csv(REPORTS / "tables" / "feature_comparison.csv"))
display(pd.read_csv(REPORTS / "validation_metrics.csv"))
display(Image(filename=str(REPORTS / "figures" / "june-performance.png")))
display(Image(filename=str(REPORTS / "figures" / "threshold-cost.png")))

## Explain the losses

A group below the 0.001 AP gain requirement is rejected and remains in the ledger. The greedy procedure is order dependent and uses logistic regression to select the representation for both model families. June is reused for multiple choices, so its best score is optimistic. No July score informed this decision.